# Building and Running an Agent Loop

An LLM, on its own, is a stateless text transformer: it reads a sequence and generates a continuation. An *agent* is different — it acts. The difference is a loop: generate → call tool → observe result → generate again. This notebook builds that loop from scratch, in about 50 lines, and then walks through the CDA library's production implementation of the same loop.

This is the keystone notebook of the series. Every subsequent notebook — context management, memory, safety, evaluation, patterns, multi-agent — assumes you understand the loop. The CDA library (`src/notebooks/agent/`) is the framework we use throughout: `Agent`, `Session`, `LLMClient`, `Config`, and the two-layer event system.

## The Think-Act-Observe Loop

Before writing a single line of code, we establish the three-step rhythm at the heart of every LLM agent.

### Pseudocode

The agent loop can be written in five lines of pseudocode:

```
while turns < max_turns:
    response = llm(messages, tools)
    if no tool_calls in response:
        return response.text       # done — model chose to stop
    for call in response.tool_calls:
        result = execute(call)
        messages.append(tool_result(result))
```

This is Anthropic's "augmented LLM" — the simplest possible agent. Three things make it an agent rather than a single LLM call:

1. **Tools:** the model can request side effects (read a file, run a command, search the web)
2. **State:** messages accumulate across turns — the model sees its own prior reasoning and tool results
3. **Autonomy:** the model decides when to stop by choosing not to call any tools

The loop terminates when the model produces a response with no `tool_calls`. This is the only termination condition (aside from the `max_turns` guard). The model signals completion by answering directly.

### Code: A 50-Line Agent

We implement the loop above as `mini_agent()` using raw `AsyncOpenAI`. No framework, no abstractions — just the protocol.

**Setup.** Imports and client initialization against OpenRouter:

In [ ]:
import os
import json
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()

client = AsyncOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url=os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
)
MODEL = "anthropic/claude-sonnet-4"

We register two simple tools — a calculator and a file writer — and define a helper that executes any tool call by name.

In [ ]:
def calculate(expression: str) -> str:
    """Evaluate a Python arithmetic expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})  # noqa: S307
        return str(result)
    except Exception as e:
        return f"Error: {e}"


def write_file(path: str, content: str) -> str:
    """Write content to a file."""
    try:
        Path(path).write_text(content)
        return f"Wrote {len(content)} bytes to {path}"
    except Exception as e:
        return f"Error: {e}"


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a Python arithmetic expression. Use for any math.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "e.g. '37 * 43'"},
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write text content to a file at the given path.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path":    {"type": "string", "description": "File path"},
                    "content": {"type": "string", "description": "Text to write"},
                },
                "required": ["path", "content"],
            },
        },
    },
]

TOOL_MAP = {"calculate": calculate, "write_file": write_file}

**Mini agent.** The `mini_agent()` function implements the loop. We print each turn's action so we can watch it think, act, and observe.

In [ ]:
async def mini_agent(task: str, max_turns: int = 10) -> str:
    """Minimal agentic loop — think, call tools, observe, repeat."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant with tools. Use tools to complete tasks."},
        {"role": "user", "content": task},
    ]

    for turn in range(max_turns):
        response = await client.chat.completions.create(    # <1>
            model=MODEL,
            messages=messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message
        print(f"[Turn {turn + 1}] finish_reason={response.choices[0].finish_reason}")

        if not msg.tool_calls:                              # <2>
            return msg.content or ""

        messages.append(msg)                               # <3>
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            print(f"  \u2192 {name}({args})")
            result = TOOL_MAP[name](**args)                # <4>
            print(f"  \u2190 {result!r}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    return "Max turns exceeded"

1. Single non-streaming call — clean and simple for demo purposes. The CDA library uses streaming (see below).
2. No tool calls → the model chose to answer directly. Return immediately.
3. The assistant message (with `tool_calls`) must be appended before the tool results. The history needs to show: assistant requested tools → tools returned results.
4. We call the Python function directly. In production the function might be a network call, a file operation, or a shell command.

We run the agent on a task that requires two tool calls in sequence.

In [ ]:
result = await mini_agent(
    "What is 37 times 43? Write the answer to /tmp/answer.txt"
)
print("\nFinal answer:", result)
print("File contents:", Path("/tmp/answer.txt").read_text())

:::{.callout-note}
The loop terminates because after writing the file, the model produces a response with no `tool_calls` — it decides the task is done. This is the only termination signal: the model stops calling tools. The `max_turns` guard is just a safety net.

:::

### Why This Needs More

The 50-line agent works, but it has gaps that matter at production scale:

- **No streaming:** The user waits for the entire response before seeing any output. A UI can't display partial results.
- **No event observability:** We can't hook into the loop for logging, monitoring, or UI rendering without modifying the loop itself.
- **No state persistence:** If the notebook kernel restarts, the conversation is lost.
- **No token tracking:** We don't know how many tokens we've used, or when we're approaching the context limit.
- **No error handling:** A rate-limit error crashes the loop. An invalid tool call silently fails.

These motivate each component of the CDA library we'll examine next.

## The Event System

The CDA library makes the agent's internal process observable through a two-layer event system. Instead of printing inside the loop, the loop *yields* structured events — and callers decide what to do with them.

### Two Layers

The two layers have different purposes:

- **`StreamEvent` / `StreamEventType`** (in `events.py`): raw LLM streaming chunks — text deltas, tool-call argument deltas, final token usage. Produced by `LLMClient`. These are noisy plumbing events.
- **`AgentEvent` / `AgentEventType`** (also in `events.py`): lifecycle events — agent started, text complete, tool invoked, agent finished. Produced by `Agent`. These are the public interface for callers.

The separation keeps concerns clean: `LLMClient` doesn't know about tools or the loop; `Agent` doesn't know about chunk parsing.

We import and inspect the event types.

In [ ]:
from notebooks.agent.events import (
    StreamEventType, StreamEvent, TextDelta, TokenUsage,
    AgentEventType, AgentEvent,
)

print("StreamEventType values:")
for e in StreamEventType:
    print(f"  {e.name}")

print("\nAgentEventType values:")
for e in AgentEventType:
    print(f"  {e.name}")

### Using LLMClient Directly

`LLMClient` wraps `AsyncOpenAI` and produces `StreamEvent` objects. We use it directly to see what the raw stream looks like.

In [ ]:
from notebooks.agent.client import LLMClient
from notebooks.agent.config import Config

config = Config()
llm = LLMClient(config)

messages = [{"role": "user", "content": "Count from 1 to 5, one number per line."}]

print("Stream events:")
async for event in llm.chat_completion(messages):
    if event.type == StreamEventType.TEXT_DELTA:
        print(f"  TEXT_DELTA: {event.text_delta.content!r}")
    elif event.type == StreamEventType.MESSAGE_COMPLETE:
        usage = event.usage
        if usage:
            print(f"  MESSAGE_COMPLETE: {usage.prompt_tokens}\u2192{usage.completion_tokens} tokens")

The `TEXT_DELTA` events arrive one token at a time. The `MESSAGE_COMPLETE` event arrives last, carrying token usage.

## The LLM Client

We walk through the key design decisions in `client.py`.

`LLMClient` is initialized lazily — the `AsyncOpenAI` client is only created on first use via `get_client()`. The main method is `chat_completion()`, which handles: building tool schemas, retrying on rate limits (exponential backoff, 3 retries), and emitting the right stream events.

The streaming implementation accumulates tool-call argument deltas by index. Tool call arguments arrive in fragments (like text), but must be assembled into a complete JSON string before the call can be executed. We walk through the key steps in `_stream_response()`:

1. Text deltas are emitted as `TEXT_DELTA` events immediately — no buffering needed.
2. Tool call deltas are accumulated in a dict keyed by `delta.index` (supports parallel tool calls from different indices).
3. When the stream ends, each accumulated tool call is emitted as a `TOOL_CALL_COMPLETE` event with fully assembled arguments.
4. The trailing `MESSAGE_COMPLETE` event carries token usage extracted from the last chunk.

The retry logic wraps the entire streaming attempt in a try/except loop. On `RateLimitError` or `APIConnectionError`, it waits $2^{\text{attempt}}$ seconds and retries up to 3 times. On other `APIError` types (bad request, invalid model, etc.), it fails immediately — retrying wouldn't help.

## The Session

The `Session` wires together all the stateful components: message history, `LLMClient`, `ToolRegistry`, and token accounting.

### Construction and State

When created from a `Config`, `Session` builds a system prompt (via `prompts.build_system_prompt()`), creates an `LLMClient`, creates a default `ToolRegistry` with all 11 builtin tools, and initializes the message history with the system message.

In [ ]:
from notebooks.agent.session import Session
from notebooks.agent.config import Config, ApprovalPolicy

config = Config(
    approval=ApprovalPolicy.YOLO,   # <1>
    allowed_tools=["read_file", "write_file", "shell"],   # <2>
)
session = Session(config)

print(f"Messages: {len(session.messages)} (system message only)")
print(f"Tools available: {[s['name'] for s in session.get_tool_schemas()]}")
print(f"Turn count: {session.turn_count}")

1. `ApprovalPolicy.YOLO` auto-approves every tool call — convenient for demos and testing.
2. `allowed_tools` restricts the registry to only these three tools, even though the full default registry has 11.

### Persistence

We run a short conversation and save it. The session stores messages, token usage, and turn count as JSON.

In [ ]:
from notebooks.agent.agent import Agent

# Run a quick two-message conversation
agent = Agent(config=config)
async for event in agent.run("What files are in the current directory?"):
    if event.type == AgentEventType.TEXT_DELTA:
        print(event.data["content"], end="", flush=True)
print()

# Save the session
save_path = agent.session.save("demo_session")
print(f"\nSaved to: {save_path}")
print(f"Turn count: {agent.session.turn_count}")

We load the session back in a fresh `Session` object and verify the messages are intact.

In [ ]:
restored = Session.load("demo_session", config)
print(f"Restored messages: {len(restored.messages)}")
print(f"Restored turn count: {restored.turn_count}")
print(f"Last message role: {restored.messages[-1]['role']}")

# Continue the conversation on the restored session
agent2 = Agent(session=restored)
async for event in agent2.run("Summarize what you found in one sentence."):
    if event.type == AgentEventType.TEXT_DELTA:
        print(event.data["content"], end="", flush=True)
print()

:::{.callout-note}
Config is NOT stored in the session file — you must provide it again when loading. This is intentional: API keys and tool configurations should come from the runtime environment, not from saved files.

:::

## The Agent Class

`Agent` is the orchestrator. It coordinates `Session`, `LLMClient`, and `ToolRegistry` into the full agentic loop.

### How `_agentic_loop()` Works

We trace through the loop annotating each stage. The structure mirrors our 50-line `mini_agent` but with streaming, event emission, and error handling added.

The loop body for each turn:

1. **Stream LLM response:** Call `session.client.chat_completion(messages, tools)` → iterate over `StreamEvent`s. Accumulate text into `accumulated_text` and tool calls into `tool_calls` list. Emit `AgentEvent.text_delta()` for each `TEXT_DELTA` stream event.
2. **Check for tool calls:** If `tool_calls` is empty, emit `AgentEvent.text_complete()` and `return` — the loop is done.
3. **Execute tools:** For each tool call, emit `AgentEvent.tool_call_start()`, call `session.registry.invoke()`, emit `AgentEvent.tool_call_complete()`, append the result to `session.messages`.
4. **Loop continues:** The model sees the tool results as the next message history. Go to step 1.

This is exactly the pseudocode from Section 1, but instrumented with events and delegating to the session and registry abstractions.

### Code: Running the Full Agent

We implement a `render_events()` helper that pretty-prints the agent's lifecycle events, then run the agent on a multi-step task.

In [ ]:
def render_events(events: list[AgentEvent]) -> None:
    """Pretty-print a list of agent lifecycle events."""
    for event in events:
        if event.type == AgentEventType.AGENT_START:
            print(f"\n\u25b6 AGENT START: {event.data['message']!r}")
        elif event.type == AgentEventType.TEXT_DELTA:
            print(event.data["content"], end="", flush=True)
        elif event.type == AgentEventType.TEXT_COMPLETE:
            print()  # newline after streaming text
        elif event.type == AgentEventType.TOOL_CALL_START:
            print(f"\n  \U0001f527 {event.data['name']}({event.data['arguments']})")
        elif event.type == AgentEventType.TOOL_CALL_COMPLETE:
            status = "\u2713" if event.data.get("success") else "\u2717"
            output = (event.data.get("output") or "")[:80]
            print(f"     {status} {output!r}")
        elif event.type == AgentEventType.AGENT_END:
            usage = event.data.get("usage") or {}
            print(f"\n\u25a0 AGENT END  tokens={usage.get('total_tokens', '?')}")
        elif event.type == AgentEventType.AGENT_ERROR:
            print(f"\n\u2717 AGENT ERROR: {event.data['error']}")

We run the agent on a task requiring 2–3 tool calls, collecting all events.

In [ ]:
from notebooks.agent.agent import Agent
from notebooks.agent.config import Config, ApprovalPolicy

run_config = Config(
    approval=ApprovalPolicy.YOLO,
    allowed_tools=["shell", "write_file", "read_file"],
)

agent = Agent(config=run_config)
events = []
async for event in agent.run(
    "Create a file /tmp/hello.txt with the content 'Hello, World!', "
    "then read it back and confirm the content."
):
    events.append(event)
    if event.type == AgentEventType.TEXT_DELTA:
        print(event.data["content"], end="", flush=True)
    elif event.type in (AgentEventType.TOOL_CALL_START, AgentEventType.AGENT_END):
        pass  # render after

print("\n\n--- Event trace ---")
render_events(events)

## Configuration

`Config` is a Pydantic model that controls all aspects of agent behavior. We survey its fields.

The key fields and their roles:

| Field | Type | Default | Purpose |
|---|---|---|---|
| `model.name` | `str` | `"anthropic/claude-sonnet-4"` | LLM to use (any OpenRouter model) |
| `model.temperature` | `float` | `1.0` | Sampling temperature |
| `model.context_window` | `int` | `200_000` | Used for context management |
| `cwd` | `Path` | `Path.cwd()` | Working directory for file tools |
| `approval` | `ApprovalPolicy` | `ON_REQUEST` | Tool approval gating |
| `max_turns` | `int` | `100` | Loop iteration limit |
| `allowed_tools` | `list[str] \| None` | `None` | Restrict available tools |
| `developer_instructions` | `str \| None` | `None` | Injected into system prompt |
| `user_instructions` | `str \| None` | `None` | Per-session customization |

The four `ApprovalPolicy` values determine which tool calls require human approval before execution. We walk through them.

In [ ]:
from notebooks.agent.config import ApprovalPolicy

for policy in ApprovalPolicy:
    print(f"  {policy.name}: {policy.value!r}")

- `YOLO`: Everything auto-approved. Use only for demos, testing, or trusted automation.
- `AUTO`: Permissive — auto-approve all except dangerous shell commands.
- `ON_REQUEST` (default): Conservative — READ tools auto-approved, write/shell/network/memory need approval.
- `NEVER`: Paranoid — approve everything manually. Useful for auditing what the agent would do without letting it act.

We cover the approval logic in depth in the Safety notebook.

Developer instructions are injected into the system prompt and let you customize the agent's behavior per-project without changing code.

In [ ]:
custom_config = Config(
    approval=ApprovalPolicy.YOLO,
    developer_instructions=(
        "You are working in a Python project that uses Ruff for linting. "
        "Always use double quotes for strings. Line length limit is 88 chars."
    ),
)
agent3 = Agent(config=custom_config)
# Print the first 500 chars of the system prompt to see instructions injected
print(agent3.session.system_prompt[:500])
print("...")

## The Full Picture

We close by sketching how the components relate, as a reference for the rest of the series.

```
Config ──────────────────────────┐
   │                             │
   ├─▶ LLMClient                 │
   │      └─ AsyncOpenAI         │
   │                             ▼
   ├─▶ ToolRegistry ────▶ Session ─────▶ Agent
   │      └─ Tool (×11)     │               │
   │                         │               │
   └─▶ ApprovalManager        │         _agentic_loop()
          (NB06)               │              │
                               └─ messages[]  │
                                  total_usage  │
                                  turn_count   │
                                  save/load ◀──┘
```

Each of the remaining notebooks zooms into one aspect of this diagram:

- **NB04:** How `Session` manages the growing `messages[]` list (context engineering)
- **NB05:** How agents remember things across sessions (memory)
- **NB06:** How `ApprovalManager` gates tool calls (safety)
- **NB07:** How to measure whether the agent is working (evaluation)
- **NB08–NB10:** Multi-agent composition and extensibility

---

■